# Traffic Fact Data - Gold Layer

## Objective
Extract and transform multimodal traffic measurements from silver.silver_multimodal to build a standardized Traffic Fact Gold Delta table (gold.gold_multimodal_fact_traffic).

## Data Flow
silver.silver_multimodal → Spark SQL / DataFrame → gold.gold_multimodal_fact_traffic

## Source
The underlying data comes from the multimodal transport network API.

## Input
Silver Delta table: silver.silver_multimodal

## Output
Gold Delta table: gold.gold_multimodal_fact_traffic

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates transactional traffic measurements, maps dimension keys, and captures user counts and trajectory attributes to maintain a performant fact table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_multimodal into PySpark.
2. **Extract Fact Data:** Query traffic measurements, generate unique measurement surrogate keys, and align foreign keys for date, time, site, and transport dimensions alongside transactional metrics.
3. **Write to Gold:** Persist processed dataset to gold.gold_multimodal_fact_traffic Delta table.

In [0]:
# Load data from silver schema
df_silver_multimodal=spark.table('workspace.silver.silver_multimodal')

In [0]:
# display the dataframe 
df_silver_multimodal.display()

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract  fact traffic Data
query_fact_traffic = """
SELECT
    ROW_NUMBER() OVER (ORDER BY timestamp, site_id) AS measurement_key,
    CAST(date_format(CAST(timestamp AS TIMESTAMP), 'yyyyMMdd') AS INT) AS date_key,
    CAST(hour(CAST(timestamp AS TIMESTAMP)) * 10000 + minute(CAST(timestamp AS TIMESTAMP)) * 100 AS INT) AS time_key,
    site_id,
    CAST(ABS(HASH(transport_mode)) AS INT) AS transport_key,
    CAST(user_count AS INT) AS user_count,
    trajectory_id,
    direction,
    lane_type,
    CAST(_ingestion_timestamp AS TIMESTAMP) AS _ingestion_timestamp
FROM workspace.silver.silver_multimodal
"""

df_fact_traffic = spark.sql(query_fact_traffic)


In [0]:
# Display df_dim_time 
df_fact_traffic.display()

# WRITING GOLD TABLE

In [0]:
df_fact_traffic\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_multimodal_fact_traffic")

# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_multimodal_fact_traffic
LIMIT 10 